# Submission to the professor's server

This notebook takes the models tested in `01_experiment.ipynb` and sends the best one to the server.

Steps:
1. Load and clean the data exactly as in `01_experiment` (same cleanup, same split, same `random_state`).
2. Compare the candidate models with 10-fold cross-validation **on the train set only**.
3. Pick the best by mean CV accuracy and evaluate it **once** on the held-out test set.
4. Retrain that model on the whole cleaned dataset.
5. Predict `abalone_app.csv` (**without** cleaning: the server expects one prediction per row, in order).
6. Send the predictions — only when `SEND = True`.

> ⚠️ The server accepts **1 submission every 12h**. Leave `SEND = False` until you are confident in the CV result.

In [1]:
import pandas as pd
import requests
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

ABALONE_DATASET_PATH = "../abalone_dataset.csv"
ABALONE_APP_PATH = "../abalone_app.csv"

URL = "https://aydanomachado.com/mlclass/03_Validation.php"
DEV_KEY = "Omicron"  # same key as Atividade 01
SEND = True                      # set to True only for the real submission

# Which model to send (must be a key of `candidates` in section 2).
# None = the best mean CV accuracy in this notebook.
#   Submission 1 (30/09 22h): "SVM (RBF) tuned (C=20, gamma=0.05)"
#   Submission 2 (01/10 10h): "Logistic Regression tuned (C=100)"
MODEL_TO_SEND = "SVM (RBF) tuned (C=20, gamma=0.05)"

RANDOM_STATE = 42

## 1. Load, clean and split (same as `01_experiment`)

In [2]:
def clean_abalone_dataset(df, percentage_threshold=10):
    """Same cleanup as 01_experiment: drop rows with inconsistent weights."""
    df_cleaned = df[
        (df["whole_weight"] >= df["shucked_weight"]) &
        (df["whole_weight"] >= df["viscera_weight"]) &
        (df["whole_weight"] >= df["shell_weight"])
    ].copy()
    sum_of_weights = df_cleaned["shucked_weight"] + df_cleaned["viscera_weight"] + df_cleaned["shell_weight"]
    percentage_difference = (sum_of_weights - df_cleaned["whole_weight"]) / df_cleaned["whole_weight"] * 100
    return df_cleaned[percentage_difference <= percentage_threshold].copy()


abalone_df = pd.read_csv(ABALONE_DATASET_PATH)
cleaned_abalone_df = clean_abalone_dataset(abalone_df)

train_set, test_set = train_test_split(
    cleaned_abalone_df, test_size=0.2, random_state=RANDOM_STATE, stratify=cleaned_abalone_df["type"]
)

X_train, y_train = train_set.drop(columns=["type"]), train_set["type"]
X_test, y_test = test_set.drop(columns=["type"]), test_set["type"]

print(f"Original: {len(abalone_df)} | cleaned: {len(cleaned_abalone_df)} | train: {len(train_set)} | test: {len(test_set)}")

Original: 3132 | cleaned: 3103 | train: 2482 | test: 621


## 2. Compare models with cross-validation

The preprocessing goes **inside** a `Pipeline`, so in each CV fold the `StandardScaler` is fitted only on that fold's
training part (in `01_experiment` it was fitted on the whole train set before the CV, which leaks a little information).
The same pipeline is later applied to `abalone_app`, so it gets exactly the same transformation.

In [3]:
num_attributes = ["length", "diameter", "height", "whole_weight", "shucked_weight", "viscera_weight", "shell_weight"]
cat_attributes = ["sex"]


def make_pipeline(model):
    preprocessor = ColumnTransformer([
        ("num", StandardScaler(), num_attributes),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_attributes),
    ])
    return Pipeline([("preprocessor", preprocessor), ("model", model)])


# Models tested in 01_experiment (default and tuned) + SVM. Add or remove candidates here.
# Tuned hyperparameters:
#   - Logistic Regression C=100 and Random Forest (500 trees, max_features=6): from the
#     GridSearch / RandomizedSearch in 01_experiment.
#   - SVM C=20, gamma=0.05: GridSearchCV on the train set over
#     C in [0.5, 1, 2, 5, 10, 20, 50] x gamma in ["scale", 0.02, 0.05, 0.1, 0.2];
#     the top configs were then compared with 10-fold CV repeated 3x.
candidates = {
    "SVM (RBF) tuned (C=20, gamma=0.05)": SVC(C=20, gamma=0.05, random_state=RANDOM_STATE),
    "Logistic Regression tuned (C=100)": LogisticRegression(C=100, max_iter=1000),
    "Random Forest tuned (500, max_features=6)": RandomForestClassifier(
        n_estimators=500, max_features=6, random_state=RANDOM_STATE, n_jobs=-1),
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    "SVM (RBF)": SVC(random_state=RANDOM_STATE),
}

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

results = []
for name, model in candidates.items():
    scores = cross_val_score(make_pipeline(model), X_train, y_train, cv=cv, scoring="accuracy")
    results.append({"model": name, "cv_mean": scores.mean(), "cv_std": scores.std()})

results_df = pd.DataFrame(results).sort_values("cv_mean", ascending=False).reset_index(drop=True)
results_df

,model,cv_mean,cv_std
0,"SVM (RBF) tuned (C=20, gamma=0.05)",0.668795,0.021155
1,Logistic Regression tuned (C=100),0.659528,0.023311
2,Logistic Regression,0.657116,0.024624
3,SVM (RBF),0.652667,0.026058
4,"Random Forest tuned (500, max_features=6)",0.651480,0.024388
5,Random Forest,0.647445,0.029703
6,KNN (k=5),0.606358,0.027743
7,Decision Tree,0.572074,0.040002


## 3. Evaluate the chosen model (`MODEL_TO_SEND`) once on the test set

In [4]:
best_name = MODEL_TO_SEND or results_df.loc[0, "model"]

best_pipeline = make_pipeline(candidates[best_name])
best_pipeline.fit(X_train, y_train)
test_predictions = best_pipeline.predict(X_test)

print(f"Chosen model: {best_name}")
print(f"CV accuracy (train): {results_df.loc[results_df['model'] == best_name, 'cv_mean'].item():.4f}")
print(f"Test set accuracy:   {accuracy_score(y_test, test_predictions):.4f}\n")
print(classification_report(y_test, test_predictions))
print(confusion_matrix(y_test, test_predictions))

Chosen model: SVM (RBF) tuned (C=20, gamma=0.05)
CV accuracy (train): 0.6688
Test set accuracy:   0.6554

              precision    recall  f1-score   support

           1       0.79      0.73      0.76       211
           2       0.51      0.58      0.54       200
           3       0.69      0.65      0.67       210

    accuracy                           0.66       621
   macro avg       0.66      0.65      0.66       621
weighted avg       0.67      0.66      0.66       621

[[154  50   7]
 [ 29 116  55]
 [ 11  62 137]]


## 4. Retrain on the whole cleaned dataset and predict `abalone_app`

The test set already did its job (an unbiased estimate), so the final model is trained with all the cleaned data.
`abalone_app` is **not** cleaned: the server needs exactly one prediction per row, in the original order.

In [5]:
X_full = cleaned_abalone_df.drop(columns=["type"])
y_full = cleaned_abalone_df["type"]

final_pipeline = make_pipeline(candidates[best_name])
final_pipeline.fit(X_full, y_full)

abalone_app_df = pd.read_csv(ABALONE_APP_PATH)
app_predictions = final_pipeline.predict(abalone_app_df[num_attributes + cat_attributes])

assert len(app_predictions) == len(abalone_app_df), "Must have one prediction per row of abalone_app"
print(f"Model: {best_name} | {len(app_predictions)} predictions")
print(pd.Series(app_predictions).value_counts().sort_index())

Model: SVM (RBF) tuned (C=20, gamma=0.05) | 1045 predictions
1    294
2    374
3    377
Name: count, dtype: int64


## 5. Send to the server (1 submission every 12h)

In [6]:
if SEND:
    payload = {
        "dev_key": DEV_KEY,
        "predictions": pd.Series(app_predictions).to_json(orient="values"),
    }
    response = requests.post(url=URL, data=payload)
    print(f" - Sent: {best_name}")
    print(" - Server response:\n", response.text)
else:
    print("SEND = False: nothing was sent. Set DEV_KEY and SEND = True in the first cell to submit.")

 - Sent: SVM (RBF) tuned (C=20, gamma=0.05)
 - Server response:
 {"status":"success","dev_key":"Omicron","accuracy":0.662200956937799,"old_accuracy":0.6622009569378}
